# VeritasVigil — The Truth Watchman

## 1. Data Loading

We load **both** `Fake.csv` and `True.csv`, assign binary labels (`0 = Fake`, `1 = Real`), and merge them into a single DataFrame. This is the correct target for real-vs-fake classification.

In [ ]:
import pandas as pd
import re
from google.colab import files

# Upload both CSVs when prompted
uploaded = files.upload()

# Load fake and real news
df_fake = pd.read_csv('Fake.csv')
df_real = pd.read_csv('True.csv')

# Assign binary labels: 0 = Fake, 1 = Real
df_fake['label'] = 0
df_real['label'] = 1

# Merge and shuffle
df = pd.concat([df_fake, df_real], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Dataset shape: {df.shape}")
print(f"Label distribution:\n{df['label'].value_counts()}")
print(df.head(3))


## 2. Custom Tokenizer

Our tokenizer performs the following steps in order:

1. **Lowercasing** — normalises case so `Trump` and `trump` are the same token.
2. **Contraction expansion** — replaces `isn't` → `is not` so negations are preserved as two tokens rather than lost.
3. **Punctuation stripping** — removes non-alphanumeric characters (preserving the `<REPEAT:n>` markers we inject in step 4).
4. **Repeated-character normalisation** — collapses elongated words like `sooooo` → `so <REPEAT:5>`. The repeat count is the number of *extra* characters beyond the first occurrence (e.g. `ooooo` has 1 base + 4 extras → `<REPEAT:4>`). This is important for fake news, which often uses informal exaggeration.

**Design note:** We keep `<REPEAT:n>` as a real token so classifiers can learn that heavy elongation correlates with sensationalist / fake content.

In [ ]:
# ── Custom Tokenizer ──────────────────────────────────────────────────────────
import re

contractions = {
    "isn't": "is not", "can't": "can not", "i'm": "i am",
    "you're": "you are", "it's": "it is", "won't": "will not",
    "didn't": "did not", "don't": "do not", "shouldn't": "should not",
    "aren't": "are not", "ain't": "am not", "we're": "we are",
    "they're": "they are", "he's": "he is", "she's": "she is",
    "that's": "that is", "there's": "there is", "i've": "i have",
    "we've": "we have", "i'd": "i would", "couldn't": "could not",
    "wouldn't": "would not", "hasn't": "has not", "hadn't": "had not",
    "doesn't": "does not", "wasn't": "was not", "weren't": "were not"
}

def expand_contractions(text):
    """Replace contractions with their full forms."""
    for key, val in contractions.items():
        text = text.replace(key, val)
    return text

def normalize_repeats(word):
    """
    Collapse elongated characters and append a <REPEAT:n> marker.
    e.g. 'sooooo' -> 'so <REPEAT:4>'  (4 extra 'o's beyond the first)
    The regex matches any character repeated 2+ extra times (3+ total).
    """
    def replacer(m):
        char = m.group(1)
        repeat_count = len(m.group(0)) - 1   # extra repetitions
        return f"{char} <REPEAT:{repeat_count}>"
    return re.sub(r'(.)\1{2,}', replacer, word)

def custom_tokenizer(text):
    """Full tokenization pipeline."""
    if not isinstance(text, str):
        return []
    # Step 1: lowercase
    text = text.lower()
    # Step 2: expand contractions
    text = expand_contractions(text)
    # Step 3: strip punctuation but keep alphanumeric + REPEAT marker characters
    text = re.sub(r'[^a-z0-9\s<>:]', '', text)
    # Step 4: split and apply repeat normalization per token
    tokens = []
    for word in text.split():
        tokens.extend(normalize_repeats(word).split())
    return tokens

# Quick sanity check
print(custom_tokenizer("This is sooooo scary and i'm NOT buying it!!!"))


## 3. Rule-Based POS Tagger

We assign coarse POS tags using two strategies:

- **Lookup lists** for auxiliary verbs (`is`, `are`, `was` …) and negations.
- **Suffix rules** for adjectives (`-ing`, `-ful`, `-ous`, `-less`, …) and for verb forms ending in `-ing`/`-ed`.
- **Verb base-form list** to distinguish `runs` (VERB, because `run` ∈ base_verbs) from `cats` (NOUN).
- **Default fallback** → NOUN (most tokens in news headlines are nouns/proper nouns).

**Note:** Suffix rules have known overlaps (e.g. `-ed` can be VERB or ADJ). We resolve this by checking VERB rules first, then ADJ, so `alarmed` → VERB, which is conservative but avoids bad lemmatization.

In [ ]:
# ── Mini POS Tagger ───────────────────────────────────────────────────────────

base_verbs = [
    "accept", "allow", "ask", "be", "become", "begin", "believe", "break", "bring", "build",
    "buy", "call", "can", "change", "choose", "clean", "come", "consider", "continue", "cook",
    "cut", "dance", "decide", "do", "draw", "drink", "drive", "eat", "explain", "fall", "feel",
    "find", "finish", "fly", "follow", "forget", "forgive", "get", "give", "go", "grow", "happen",
    "have", "hear", "help", "hold", "hope", "include", "invite", "jump", "keep", "know", "laugh",
    "learn", "leave", "like", "listen", "live", "look", "lose", "love", "make", "mean", "meet",
    "move", "need", "open", "pay", "play", "prefer", "prepare", "pull", "push", "put", "rain",
    "read", "remember", "run", "say", "see", "seem", "sell", "send", "show", "sing", "sit", "sleep",
    "smile", "speak", "spend", "stand", "start", "stay", "stop", "study", "succeed", "swim",
    "take", "talk", "teach", "tell", "think", "throw", "touch", "travel", "try", "turn", "understand",
    "use", "wait", "walk", "want", "watch", "win", "work", "write", "admit", "announce", "arrive",
    "attend", "attract", "avoid", "beat", "behave", "blame", "breathe", "burn", "calculate", "care",
    "cheat", "climb", "complain", "concentrate", "connect", "control", "convince", "correct", "cost",
    "cover", "create", "cross", "damage", "delay", "deliver", "depend", "describe", "destroy",
    "develop", "discover", "divide", "doubt", "dress", "educate", "employ", "encourage", "enjoy",
    "escape", "estimate", "examine", "exist", "expect", "experience", "express", "face", "feed",
    "fight", "fill", "fix", "fold", "forbid", "freeze", "gather", "guess", "handle", "hide", "identify",
    "imagine", "improve", "increase", "influence", "inform", "insist", "inspire", "introduce",
    "join", "joke", "kick", "kill", "knock", "lead", "lend", "limit", "maintain", "manage",
    "mark", "marry", "mention", "miss", "notice", "obey", "observe", "occur", "offer", "organize",
    "pack", "participate", "pass", "perform", "persuade", "point", "postpone", "promise",
    "protect", "prove", "react", "realize", "receive", "recommend", "reduce", "refuse", "regret",
    "relax", "remove", "repair", "replace", "reply", "report", "rescue", "respect", "respond",
    "return", "save", "scream", "search", "select", "separate", "shout", "sign", "solve", "sort",
    "suggest", "support", "surprise", "survive", "suspect", "switch", "test",
    "train", "translate", "treat", "trust", "update", "value",
    "visit", "warn", "wish", "wonder", "worry"
]

def mini_pos_tagger(token):
    """Assign a coarse POS tag using suffix rules and lookup lists."""
    token = token.lower()

    # Skip REPEAT markers — tag them as special
    if token.startswith('<repeat:') and token.endswith('>'):
        return "REPEAT"

    # Auxiliary verbs lookup
    aux_verbs = {"is", "am", "are", "was", "were", "be", "been", "have", "has", "do", "did",
                 "will", "would", "could", "should", "may", "might", "shall", "must"}
    if token in aux_verbs:
        return "VERB"

    # Negation
    if token in {"no", "not", "never", "none"}:
        return "NEG"

    # Verb: -ing and -ed endings (before ADJ to prioritise VERB)
    if token.endswith("ing") and len(token) > 4:
        return "VERB"
    if token.endswith("ed") and len(token) > 3:
        return "VERB"

    # Adjective suffixes
    adj_suffixes = ("able", "ible", "al", "ant", "ent", "ary", "en",
                    "ful", "ic", "ical", "ish", "ive", "less", "ous",
                    "eous", "ious", "some", "y")
    if token.endswith(adj_suffixes):
        return "ADJ"

    # -s ending: check if stem is a known base verb
    if token.endswith("s") and len(token) > 3:
        stem = token[:-1]
        if stem in base_verbs:
            return "VERB"
        return "NOUN"

    return "NOUN"

def tag_tokens(tokens):
    return [(token, mini_pos_tagger(token)) for token in tokens]

# Quick sanity check
sample = custom_tokenizer("The president is running sooooo fast")
print(tag_tokens(sample))


In [ ]:
# Apply tokenizer and POS tagger to the dataset
df['tokens']   = df['title'].apply(custom_tokenizer)
df['pos_tags'] = df['tokens'].apply(tag_tokens)

print(df[['title', 'tokens', 'pos_tags', 'label']].head(3))


## 4. POS-Guided Lemmatizer

Rules are applied **only when the POS tag matches**, preventing incorrect reduction (e.g. `news` → `new` is avoided because `news` is tagged NOUN but the rule checks length guard).

| POS | Rule | Example |
|-----|------|---------|
| VERB | strip `-ing` | `running` → `run` |
| VERB | strip `-ed` | `claimed` → `claim` |
| VERB | strip `-s` | `runs` → `run` |
| NOUN | strip `-s` | `claims` → `claim` |
| ADJ | strip `-ful`/`-ous`/`-able` (3 chars) | `powerful` → `power` |
| ADJ | strip `-ible`/`-ical`/`-less`/`-eous`/`-ious`/`-some` (4 chars) | `logical` → `log` |
| ADV | strip `-ly` | `quickly` → `quick` |
| REPEAT/NEG | pass through unchanged | `<REPEAT:4>` stays |

All rules include minimum-length guards to avoid over-stripping short words.

In [ ]:
# ── POS-Guided Lemmatizer ────────────────────────────────────────────────────

def simple_lemmatizer(token, pos):
    """Return the base form of a token, guided by its POS tag."""
    token = token.lower()

    # Pass REPEAT markers and negations through unchanged
    if pos in ("REPEAT", "NEG"):
        return token

    if pos == "VERB":
        if token.endswith("ing") and len(token) > 4:
            return token[:-3]                    # running → run
        elif token.endswith("ed") and len(token) > 3:
            return token[:-2]                    # claimed → claim
        elif token.endswith("s") and len(token) > 3:
            return token[:-1]                    # runs    → run

    elif pos == "NOUN":
        if token.endswith("s") and len(token) > 3:
            return token[:-1]                    # claims  → claim

    elif pos == "ADV":
        if token.endswith("ly") and len(token) > 4:
            return token[:-2]                    # quickly → quick

    elif pos == "ADJ":
        # 3-char suffixes first
        if token.endswith("ful") and len(token) > 5:
            return token[:-3]                    # powerful → power
        elif token.endswith("ous") and len(token) > 5:
            return token[:-3]                    # famous   → fam
        elif token.endswith("able") and len(token) > 6:
            return token[:-4]                    # capable  → cap
        # 4-char suffixes
        elif token.endswith("ible") and len(token) > 6:
            return token[:-4]                    # possible → poss
        elif token.endswith("ical") and len(token) > 6:
            return token[:-4]                    # logical  → log
        elif token.endswith("less") and len(token) > 6:
            return token[:-4]                    # hopeless → hope
        elif token.endswith("eous") and len(token) > 6:
            return token[:-4]                    # gorgeous → gorg
        elif token.endswith("ious") and len(token) > 6:
            return token[:-4]                    # serious  → seri
        elif token.endswith("some") and len(token) > 6:
            return token[:-4]                    # awesome  → awe

    return token

# Apply to dataframe
df['lemmatized_tokens'] = df['pos_tags'].apply(
    lambda tag_list: [simple_lemmatizer(w, p) for w, p in tag_list]
)

# Quick check
print(df[['title', 'lemmatized_tokens', 'label']].head(3))


## 5. Feature Extraction and Classification

We extract features two ways:
- **Bag-of-Words (BoW)** via `CountVectorizer` — raw term counts.
- **TF-IDF** via `TfidfVectorizer` — down-weights very common words.

We then train two classifiers:
1. **Naïve Bayes** on BoW — fast, probabilistic, good baseline for text.
2. **SVM (linear kernel)** on TF-IDF — typically the strongest shallow classifier for text.

Target: `label` column (0 = Fake, 1 = Real) — the correct binary fake-vs-real task.

In [ ]:
# ── Feature Extraction & Classification ──────────────────────────────────────
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.metrics import classification_report

# Join lemmatized tokens back to string
df['clean_text'] = df['lemmatized_tokens'].apply(lambda tokens: ' '.join(tokens))

# Feature matrices
bow_vectorizer   = CountVectorizer()
X_bow   = bow_vectorizer.fit_transform(df['clean_text'])

tfidf_vectorizer = TfidfVectorizer()
X_tfidf = tfidf_vectorizer.fit_transform(df['clean_text'])

# Correct binary target: Fake(0) vs Real(1)
y = df['label']

# Train / test splits
X_train_bow,   X_test_bow,   y_train, y_test = train_test_split(X_bow,   y, test_size=0.2, random_state=42)
X_train_tfidf, X_test_tfidf, _,       _      = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)

# ── Naïve Bayes on BoW ────────────────────────────────────────────────────────
nb_bow = MultinomialNB()
nb_bow.fit(X_train_bow, y_train)
pred_nb = nb_bow.predict(X_test_bow)

print("=" * 50)
print("Naïve Bayes (Bag-of-Words)")
print("=" * 50)
print(classification_report(y_test, pred_nb, target_names=['Fake', 'Real']))

# ── SVM on TF-IDF ─────────────────────────────────────────────────────────────
svm_tfidf = SVC(kernel='linear', probability=True)
svm_tfidf.fit(X_train_tfidf, y_train)
pred_svm = svm_tfidf.predict(X_test_tfidf)

print("=" * 50)
print("SVM (TF-IDF)")
print("=" * 50)
print(classification_report(y_test, pred_svm, target_names=['Fake', 'Real']))


## 6. Visualizations

### 6a. Token Frequency Distribution

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt

# Filter out REPEAT markers from word freq (handled separately below)
all_tokens = [t for t in ' '.join(df['clean_text']).split() if not t.startswith('<repeat:')]
word_counts = Counter(all_tokens)
common_words = word_counts.most_common(20)

words, counts = zip(*common_words)
plt.figure(figsize=(12, 5))
plt.bar(words, counts, color='steelblue')
plt.xticks(rotation=45, ha='right')
plt.title('Top 20 Most Frequent Tokens (excluding REPEAT markers)')
plt.xlabel('Token')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()


### 6b. Word Cloud

In [ ]:
from wordcloud import WordCloud

text = ' '.join(df['clean_text'])
wordcloud = WordCloud(width=800, height=400, background_color='white',
                      regexp=r'[a-z]+').generate(text)

plt.figure(figsize=(12, 6))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud of Lemmatized Tokens')
plt.tight_layout()
plt.show()


### 6c. REPEAT Token Distribution

This shows how often each elongation level (REPEAT:2, REPEAT:3 …) appears across all headlines. Fake news is expected to have more high-repeat tokens due to sensationalist writing.

In [ ]:
# Extract all <REPEAT:n> tokens and count by n value
import re

repeat_tokens = [t for t in ' '.join(df['clean_text']).split() if t.startswith('<repeat:')]
repeat_counts = Counter(repeat_tokens)

if repeat_counts:
    labels_r = sorted(repeat_counts.keys(), key=lambda x: int(re.search(r'\d+', x).group()))
    values_r = [repeat_counts[l] for l in labels_r]

    plt.figure(figsize=(10, 4))
    plt.bar(labels_r, values_r, color='tomato')
    plt.xticks(rotation=45, ha='right')
    plt.title('Distribution of <REPEAT:n> Tokens')
    plt.xlabel('Repeat Token')
    plt.ylabel('Frequency')
    plt.tight_layout()
    plt.show()
    print(f"Total REPEAT tokens found: {sum(values_r)}")
else:
    print("No <REPEAT:n> tokens found in dataset (headlines may not contain elongated words).")


### 6d. Confusion Matrices

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Naïve Bayes
cm_nb = confusion_matrix(y_test, pred_nb)
sns.heatmap(cm_nb, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Fake', 'Real'], yticklabels=['Fake', 'Real'])
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].set_title('Confusion Matrix — Naïve Bayes (BoW)')

# SVM
cm_svm = confusion_matrix(y_test, pred_svm)
sns.heatmap(cm_svm, annot=True, fmt='d', cmap='Greens', ax=axes[1],
            xticklabels=['Fake', 'Real'], yticklabels=['Fake', 'Real'])
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')
axes[1].set_title('Confusion Matrix — SVM (TF-IDF)')

plt.tight_layout()
plt.show()


### 6e. ROC Curves

In [ ]:
from sklearn.metrics import roc_curve, auc

# Naïve Bayes probabilities
nb_probs  = nb_bow.predict_proba(X_test_bow)[:, 1]
fpr_nb, tpr_nb, _ = roc_curve(y_test, nb_probs)
roc_auc_nb = auc(fpr_nb, tpr_nb)

# SVM probabilities (requires probability=True, set above)
svm_probs = svm_tfidf.predict_proba(X_test_tfidf)[:, 1]
fpr_svm, tpr_svm, _ = roc_curve(y_test, svm_probs)
roc_auc_svm = auc(fpr_svm, tpr_svm)

plt.figure(figsize=(8, 6))
plt.plot(fpr_nb,  tpr_nb,  label=f'Naïve Bayes  (AUC = {roc_auc_nb:.3f})',  color='steelblue', lw=2)
plt.plot(fpr_svm, tpr_svm, label=f'SVM          (AUC = {roc_auc_svm:.3f})', color='darkorange', lw=2)
plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Random classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves — Naïve Bayes vs SVM')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 7. Final Report

### Tokenizer Design
The custom tokenizer processes text in four stages. Lowercasing ensures uniform representation. Contraction expansion (25 contractions) ensures that negations like `won't` are preserved as `will not` rather than collapsed into a single opaque token. Punctuation stripping removes noise while preserving the special `<REPEAT:n>` markers injected in the final stage. The repeated-character normalizer detects any character repeated 3 or more times consecutively (regex `(.)\1{2,}`) and replaces the run with the base character plus a marker encoding the number of extra repetitions. This converts `sooooo` → `so <REPEAT:4>`, preserving the signal that the author was using informal exaggeration.

### POS Tagger Design
The mini POS tagger uses two mechanisms. First, small hand-crafted lookup sets handle auxiliary verbs and negations with 100% precision on known forms. Second, suffix rules handle open-vocabulary words: `-ing`/`-ed` map to VERB, a set of 18 suffixes map to ADJ, and an `-s` ending is resolved against the base-verb list before defaulting to NOUN. The overall default is NOUN, consistent with the high proportion of nouns in news headlines.

### Lemmatizer Design
Lemmatization rules are gated by POS tag, which prevents the most common errors. For example, the noun `press` would incorrectly be stripped to `pres` by a POS-unaware stemmer, but our tagger labels it NOUN and the NOUN rule (strip `-s`) is only applied when length > 3, so `press` (length 5) → `pres` — this is still imperfect, demonstrating a known limitation of pure suffix-based lemmatization without a dictionary.

### Impact of Normalisation
Repeated-character normalisation converts rare out-of-vocabulary elongated forms into vocabulary-present base tokens plus frequency-informative REPEAT markers. This reduces sparsity and may add a signal absent from the raw text. POS-guided lemmatization further reduces sparsity by collapsing inflected forms, which is especially useful for BoW where `run`, `running`, and `runs` would otherwise be three separate features.

### Classifier Comparison
Both classifiers are trained on lemmatized text. Naïve Bayes on BoW is a strong baseline due to the near-independence of headline words. SVM with TF-IDF typically outperforms it because TF-IDF penalises tokens that appear in virtually every document (stop words), and SVM's margin-maximisation is robust to the high-dimensional sparse feature space. The ROC curves and AUC scores above quantify this comparison precisely for our specific dataset split.

### Brief Comparison with Off-the-Shelf Pipelines
Libraries like spaCy or NLTK provide dictionary-backed lemmatizers and Penn Treebank POS taggers trained on large corpora. These would produce more accurate POS tags (distinguishing VBG/VBN/VBD rather than a single VERB) and handle irregular forms (e.g. `went` → `go`). Our custom pipeline trades accuracy for transparency and explainability — every rule is visible and documented — which meets the assignment's requirement of originality over borrowed magic.